# USD FX Correlation Dashboard

Run this notebook from the `Correlation` folder. It uses the same code as `build_correlation_dashboard.py`, then displays the generated HTML dashboard inline.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import IFrame, display

from build_correlation_dashboard import (
    DEFAULT_EXCHANGE,
    DEFAULT_LT_WINDOW,
    DEFAULT_ST_WINDOW,
    EXTRA_INSTRUMENTS,
    USD_CURRENCIES,
    calculate_log_returns,
    calculate_window_correlation,
    fetch_all,
    load_existing_output,
    order_frame_columns,
    order_matrix,
    write_excel,
    write_html,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

## Parameters

In [ ]:
EXCHANGE = DEFAULT_EXCHANGE
BARS = 250
ST_WINDOW = DEFAULT_ST_WINDOW
LT_WINDOW = DEFAULT_LT_WINDOW
OUTPUT_DIR = Path("output")

# Set this to True when you only want to rebuild the report from existing downloaded prices.
USE_EXISTING_OUTPUT = False

## Retrieve Or Load Prices

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
excel_path = OUTPUT_DIR / "usd_fx_correlation.xlsx"
html_path = OUTPUT_DIR / "usd_fx_correlation.html"

if USE_EXISTING_OUTPUT:
    prices, metadata = load_existing_output(excel_path)
else:
    prices, metadata = fetch_all(USD_CURRENCIES, EXTRA_INSTRUMENTS, EXCHANGE, BARS)

prices = order_frame_columns(prices)
metadata

## Calculate Log Returns And Correlations

In [ ]:
log_returns = order_frame_columns(calculate_log_returns(prices))

corr_st = order_matrix(calculate_window_correlation(log_returns, ST_WINDOW))
corr_lt = order_matrix(calculate_window_correlation(log_returns, LT_WINDOW))
corr_diff = corr_st - corr_lt

print(f"Price rows: {len(prices):,}")
print(f"Return rows: {len(log_returns):,}")
print(f"Symbols: {len(corr_diff.columns):,}")
off_diagonal = ~np.eye(len(corr_diff), dtype=bool)
print(f"Max abs ST-LT diff: {corr_diff.where(off_diagonal).abs().stack().max():.4f}")

## DXY Reference Strip

In [ ]:
dxy_view = pd.DataFrame({
    "ST_corr_vs_DXY": corr_st.loc["DXY"],
    "LT_corr_vs_DXY": corr_lt.loc["DXY"],
    "diff_ST_minus_LT": corr_diff.loc["DXY"],
}).drop(index="DXY")

dxy_view.sort_values("diff_ST_minus_LT", ascending=False).head(20)

## Write Excel And HTML Outputs

In [ ]:
write_excel(
    excel_path,
    prices,
    log_returns,
    corr_st,
    corr_lt,
    corr_diff,
    metadata,
    ST_WINDOW,
    LT_WINDOW,
)

write_html(
    html_path,
    prices,
    log_returns,
    corr_st,
    corr_lt,
    corr_diff,
    metadata,
    ST_WINDOW,
    LT_WINDOW,
)

print(f"Wrote {excel_path}")
print(f"Wrote {html_path}")

## Display Dashboard

In [ ]:
display(IFrame(src=html_path.as_posix(), width="100%", height=900))